<a href="https://colab.research.google.com/github/steve-dev-55/Mboablog/blob/main/import_dataset_drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import userdata
import os

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')

# Pas besoin de kaggle.json du tout
print('✅ Kaggle configuré via secrets')

✅ Kaggle configuré via secrets


In [7]:
from google.colab import drive, userdata
from pathlib import Path
import subprocess, shutil, os

# ── 1. Monte Drive ────────────────────────────────────────────
drive.mount('/content/drive')

DRIVE_DATASET = Path('/content/drive/MyDrive/nightclub_dataset')
DRIVE_DATASET.mkdir(parents=True, exist_ok=True)

# ── 2. Config Kaggle via Secrets Colab ────────────────────────
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')

# Crée kaggle.json pour la lib kaggle
import json
KAGGLE_DIR = Path('/root/.kaggle')
KAGGLE_DIR.mkdir(exist_ok=True)
(KAGGLE_DIR / 'kaggle.json').write_text(json.dumps({
    'username': os.environ['KAGGLE_USERNAME'],
    'key':      os.environ['KAGGLE_KEY'],
}))
os.chmod(KAGGLE_DIR / 'kaggle.json', 0o600)
print('✅ Kaggle configuré')

# ── 3. Installation kaggle ────────────────────────────────────
subprocess.run('pip install -q kaggle', shell=True, capture_output=True)

# ── 4. Téléchargement datasets ────────────────────────────────
DATASETS = [
    'djoumessimbasteve/10-kwatt-nightclub-man',
    'djoumessimbasteve/10-kwatt-nightclub-girl',
    'djoumessimbasteve/kwatt-nightclub-bar',
    'djoumessimbasteve/kwatt-nightclub-exterieur',
    'djoumessimbasteve/kwatt-nightclub-office',
]

DOWNLOAD_DIR = Path('/content/downloads')
DOWNLOAD_DIR.mkdir(exist_ok=True)

for dataset in DATASETS:
    name     = dataset.split('/')[1]
    dest_dir = DRIVE_DATASET / name

    if dest_dir.exists() and len(list(dest_dir.glob('*.png'))) > 0:
        n = len(list(dest_dir.glob('*.png')))
        print(f'  ♻️  {name} déjà sur Drive ({n} images)')
        continue

    print(f'  ⬇️  {name}...')
    r = subprocess.run(
        f'kaggle datasets download -d {dataset} -p {DOWNLOAD_DIR} --unzip',
        shell=True, capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f'  ❌ {r.stderr[-200:]}')
        continue

    dest_dir.mkdir(parents=True, exist_ok=True)
    downloaded = list(DOWNLOAD_DIR.glob('**/*.png')) + list(DOWNLOAD_DIR.glob('**/*.jpg'))
    for img in downloaded:
        shutil.copy2(img, dest_dir / img.name)

    shutil.rmtree(DOWNLOAD_DIR)
    DOWNLOAD_DIR.mkdir(exist_ok=True)

    n = len(list(dest_dir.glob('*.png')))
    print(f'  ✅ {name} → Drive ({n} images)')

# ── 5. Récap ──────────────────────────────────────────────────
print('\n📋 DATASETS SUR DRIVE')
print('─' * 45)
for dataset in DATASETS:
    name = dataset.split('/')[1]
    dest = DRIVE_DATASET / name
    n    = len(list(dest.glob('*.png'))) if dest.exists() else 0
    print(f'  {"✅" if n > 0 else "❌"} {name:<35} {n} imgs')
print(f'\n  Dossier : {DRIVE_DATASET}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Kaggle configuré
  ⬇️  10-kwatt-nightclub-man...
  ✅ 10-kwatt-nightclub-man → Drive (30 images)
  ⬇️  10-kwatt-nightclub-girl...
  ✅ 10-kwatt-nightclub-girl → Drive (31 images)
  ⬇️  kwatt-nightclub-bar...
  ✅ kwatt-nightclub-bar → Drive (60 images)
  ⬇️  kwatt-nightclub-exterieur...
  ✅ kwatt-nightclub-exterieur → Drive (60 images)
  ⬇️  kwatt-nightclub-office...
  ✅ kwatt-nightclub-office → Drive (33 images)

📋 DATASETS SUR DRIVE
─────────────────────────────────────────────
  ✅ 10-kwatt-nightclub-man              30 imgs
  ✅ 10-kwatt-nightclub-girl             31 imgs
  ✅ kwatt-nightclub-bar                 60 imgs
  ✅ kwatt-nightclub-exterieur           60 imgs
  ✅ kwatt-nightclub-office              33 imgs

  Dossier : /content/drive/MyDrive/nightclub_dataset


In [6]:
from google.colab import userdata
from pathlib import Path
import subprocess, os, json, shutil

# Config Kaggle
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')

KAGGLE_DIR = Path('/root/.kaggle')
KAGGLE_DIR.mkdir(exist_ok=True)
(KAGGLE_DIR / 'kaggle.json').write_text(json.dumps({
    'username': os.environ['KAGGLE_USERNAME'],
    'key':      os.environ['KAGGLE_KEY'],
}))
os.chmod(KAGGLE_DIR / 'kaggle.json', 0o600)

# Test sur UN seul dataset avec erreur complète visible
r = subprocess.run(
    'kaggle datasets download -d djoumessimbasteve/10-kwatt-nightclub-man '
    '-p /content/downloads --unzip',
    shell=True, capture_output=True, text=True
)
print('STDOUT:', r.stdout)
print('STDERR:', r.stderr)
print('CODE  :', r.returncode)

STDOUT: 403 Client Error: Forbidden for url: https://www.kaggle.com/api/v1/datasets/metadata/djoumessimbasteve/10-kwatt-nightclub-man

STDERR: 
CODE  : 1
